<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/02-numpy.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 2 — NumPy: Thinking in Arrays

Companion notebook to [NumPy: Thinking in Arrays](https://www.ai.biz/books/python-primer/numpy/).

Three things cause most numerical bugs: not knowing what is in memory, not knowing whether you
hold a view or a copy, and not knowing which axis is collapsing. Run every cell.


In [ ]:
import numpy as np
print('numpy', np.__version__)


## 1. Anatomy of an array

Shape and strides are the only things making a flat buffer multidimensional.


In [ ]:
a = np.arange(12).reshape(3, 4)
print(a)
print()
print('shape   ', a.shape)
print('ndim    ', a.ndim)
print('dtype   ', a.dtype)
print('strides ', a.strides, ' <- bytes to step along each axis')
print('nbytes  ', a.nbytes)


In [ ]:
# The buffer is flat. reshape just rewrites the header.
print('flat view:', a.ravel())
print('reshaped (2,6):')
print(a.reshape(2, 6))


## 2. dtype has consequences

Fixed-width integers overflow silently. This is not a bug, it is what fixed width means.


In [ ]:
x = np.array([127], dtype=np.int8)
print('127 + 1 as int8 =', x + 1, ' <- wrapped, no warning')

print()
print('0.1 + 0.2 == 0.3        ->', 0.1 + 0.2 == 0.3)
print('np.isclose(0.1+0.2,0.3) ->', np.isclose(0.1 + 0.2, 0.3))


In [ ]:
# float32 halves memory and loses precision. Watch it drift.
big = np.ones(10_000_000, dtype=np.float32)
print('float32 sum of 10M ones:', big.sum())
print('float64 sum of 10M ones:', big.astype(np.float64).sum())


## 3. Views and copies — the bug you will write


In [ ]:
a = np.arange(10)
b = a[2:5]        # basic slicing -> VIEW
b[0] = 999
print('a is now:', a)
print('b.base is a ->', b.base is a)


In [ ]:
a = np.arange(10)
c = a[a > 5]      # boolean mask -> COPY
c[0] = -1
print('a unchanged:', a)
print('c.base is None ->', c.base is None)


In [ ]:
# Rule of thumb
a = np.arange(10)
for label, sel in [('a[2:5]',   a[2:5]),
                   ('a[::2]',   a[::2]),
                   ('a[[1,3]]', a[[1, 3]]),
                   ('a[a>5]',   a[a > 5])]:
    print(f'{label:<10} -> {"VIEW" if sel.base is not None else "COPY"}')


## 4. Axes: the axis you name is the axis that disappears


In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])
print('a shape', a.shape)
print('sum(axis=0)', a.sum(axis=0), 'shape', a.sum(axis=0).shape, '<- the 2 is gone')
print('sum(axis=1)', a.sum(axis=1), 'shape', a.sum(axis=1).shape, '<- the 3 is gone')
print('sum()      ', a.sum())


In [ ]:
# keepdims retains the collapsed axis so the result broadcasts back
print('without keepdims:', a.mean(axis=0).shape)
print('with keepdims:   ', a.mean(axis=0, keepdims=True).shape)
print()
print(a - a.mean(axis=0, keepdims=True))


## 5. Broadcasting

Compare shapes **right to left**. Dimensions must be equal, or one of them must be 1.


In [ ]:
def try_broadcast(s1, s2):
    try:
        r = np.broadcast_shapes(s1, s2)
        print(f'{str(s1):<12} + {str(s2):<12} -> {r}')
    except ValueError:
        print(f'{str(s1):<12} + {str(s2):<12} -> ValueError')

try_broadcast((3, 4), (4,))
try_broadcast((3, 1), (1, 4))
try_broadcast((2, 3, 4), ())
try_broadcast((3, 4), (3,))     # the one everybody hits


In [ ]:
a = np.ones((3, 4))
b = np.array([10, 20, 30])      # shape (3,)

try:
    a + b
except ValueError as e:
    print('as expected:', e)

print()
print('fixed with a new axis:')
print(a + b[:, np.newaxis])


In [ ]:
# Broadcasting copies nothing. It uses a stride of zero.
x = np.array([1.0, 5.0, 9.0])
pairwise = x[:, None] - x[None, :]
print(pairwise)


## 6. Boolean masks


In [ ]:
a = np.array([3, 8, 1, 9, 4])
print('mask      ', a > 5)
print('values    ', a[a > 5])
print('positions ', np.where(a > 5)[0])
print('count     ', (a > 5).sum())
print('proportion', (a > 5).mean())


In [ ]:
# Use & | ~ with parentheses, never and/or/not
print(a[(a > 2) & (a < 9)])
try:
    a[a > 2 and a < 9]
except ValueError as e:
    print('and/or fails:', str(e)[:60], '...')


## 7. Missing values


In [ ]:
a = np.array([1.0, np.nan, 3.0])
print('nan == nan   ->', np.nan == np.nan, ' <- never test this way')
print('isnan        ->', np.isnan(a))
print('a.sum()      ->', a.sum(), ' <- one nan poisons it')
print('np.nansum(a) ->', np.nansum(a))
print('dropped      ->', a[~np.isnan(a)])


## 8. Worked example: standardising a feature matrix


In [ ]:
rng = np.random.default_rng(0)   # the modern generator API
X = rng.normal(loc=[10, 200, 3], scale=[2, 50, 0.5], size=(1000, 3))

mu = X.mean(axis=0)      # (3,)  one value per feature
sigma = X.std(axis=0)    # (3,)
Z = (X - mu) / sigma     # (1000,3) against (3,) aligns from the right

print('means after: ', np.round(Z.mean(axis=0), 12))
print('stds after:  ', np.round(Z.std(axis=0), 12))


## Try it yourself

1. Build a 5x5 multiplication table using only broadcasting and `np.arange`.
2. Given a 2D array, subtract each **row's** mean rather than each column's. Which axis, and do you need `keepdims`?
3. Write a function that normalises an array in place, then prove with `.base` that the caller's data changed.
4. Time `a = a + 1` against `a += 1` on a 50-million element array and explain the gap.
